In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents[1]

{'id': '226a4baf2f',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What’s new in the 2025 edition?',
 'answer': '- Deployment module updated to **FastAPI** (replacing Flask) and new tools.\n- Neural networks taught with **PyTorch** (theory videos in Keras are kept; an additional PyTorch implementation video is provided).\n- Deep learning deployment uses **ONNX Runtime** on AWS Lambda (replacing TensorFlow Lite).'}

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

118

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


## Generate Questions with Structured Output

In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json

user_prompt = json.dumps(doc)

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
result = response.output_parsed

print(result)

questions=['I just found this course late — can I still sign up and take it?', 'If I join after the course started, am I still eligible for the certificate?', 'Is it okay to start the course now even though I missed the beginning?', 'What do I need to do to get a certificate if I join late?', 'Can I still participate in the course, or is it too late to join?']


In [13]:
print(result.questions)

['I just found this course late — can I still sign up and take it?', 'If I join after the course started, am I still eligible for the certificate?', 'Is it okay to start the course now even though I missed the beginning?', 'What do I need to do to get a certificate if I join late?', 'Can I still participate in the course, or is it too late to join?']


## Reusable Utilities

In [14]:
from evaluation_utils import llm_structured

In [15]:
result, usage = llm_structured(
    client=openai_client, 
    instructions=data_gen_instructions,
    user_prompt=user_prompt,
    output_type=Questions,
    )

print(result.questions)

['I just found this course late — can I still enroll and follow along?', 'Is it okay to join the course after it already started?', 'If I’m late to the course, can I still participate and do the assignments?', 'Can I still get a certificate if I join the course now?', 'What’s the deadline for getting the certificate if I start the course late?']


## Tracking Cost

In [16]:
usage.input_tokens, usage.output_tokens

(207, 88)

In [17]:
from evaluation_utils import calc_price

In [18]:
cost = calc_price(usage)
cost

{'input_cost': 0.00015525, 'output_cost': 0.000396, 'total_cost': 0.00055125}

In [19]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })
    
records

[{'question': 'I just found this course late — can I still enroll and follow along?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to join the course after it already started?',
  'document': '74eb249bbf'},
 {'question': 'If I’m late to the course, can I still participate and do the assignments?',
  'document': '74eb249bbf'},
 {'question': 'Can I still get a certificate if I join the course now?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for getting the certificate if I start the course late?',
  'document': '74eb249bbf'}]

When evaluating search, we'll ask the search engine the generated question. 

Then we'll check if it retrieves the document with this ID

## Generating Ground Truth for all Documents

For each document:
* Convert the document to JSON so we can send it to LLM
* Ask the LLM to return a `question` object
* Create one ground truth record for each generated question 

In [20]:
# We use llm_structured_retry as if one request fails by aa temporary API or network issues,
# it waits briefly and tries again
from evaluation_utils import llm_structured_retry

In [21]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)
    
    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
    )
    
    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })
    
    return results, usage

In [22]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

## Parallel Processing

In [23]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [24]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/118 [00:00<?, ?it/s]

In [25]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

590

In [26]:
ground_truth[0]

{'question': 'I just found this course late — am I still allowed to join in now?',
 'document': '74eb249bbf'}

In [27]:
usages[0]

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=86, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=293)

In [28]:
total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.091164

In [29]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.091164

In [30]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [32]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)